In [21]:
import pandas as pd
import streamlit as st
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
import numpy as np
import tensorflow as tf
#from tensorflow.keras.models import Sequential, load_model
#from tensorflow.keras.layers import Dense
#from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import pickle
import os

In [22]:
data = pd.read_csv('churn_modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [23]:
data = data.drop(['RowNumber','CustomerId','Surname'], axis = 1)
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [24]:
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

In [25]:
one_hot_Encoder_geo = OneHotEncoder(handle_unknown='ignore')
geo_encoded = one_hot_Encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=one_hot_Encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [26]:
data = pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [27]:
X = data.drop('EstimatedSalary', axis = 1)
Y = data['EstimatedSalary']

# split the data into training and testing set
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)   

## Scale these features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [28]:
with open('sl_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

## Save the encoders and scaler for future use
with open('sl_label_enc_gender.pkl', 'wb') as f:
    pickle.dump(label_encoder_gender, f)

with open('sl_one_hot_encoder_geography.pkl', 'wb') as f:
    pickle.dump(one_hot_Encoder_geo, f)

In [29]:
## ANN with regrattion problem statement
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

In [30]:
model = Sequential([
    Dense(64, activation='relu',input_shape=(X_train.shape[1],)), # Hidden Layer 1 connected to input layer
    Dense(32, activation='relu'), # Hidden Layer 2
    Dense(1,) # Output layer with linear activation function
])

#complie the model 
model.compile(optimizer='adam',loss='mean_absolute_error',metrics=['mae'])

model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_3 (Dense)             (None, 64)                832       
                                                                 
 dense_4 (Dense)             (None, 32)                2080      
                                                                 
 dense_5 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [31]:
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard

log_dir = "reglogs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [32]:
EarlyStopping_Callback = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [33]:
history = model.fit(
    X_train, 
    Y_train, 
    validation_data=(X_test, Y_test), 
    epochs=100, 
    callbacks=[EarlyStopping_Callback, 
               tensorflow_callback]
    )

Epoch 1/100
250/250 [==============================] - 1s 2ms/step - loss: 100388.4531 - mae: 100388.4531 - val_loss: 98555.6328 - val_mae: 98555.6328
Epoch 2/100
250/250 [==============================] - 0s 1ms/step - loss: 99757.7500 - mae: 99757.7500 - val_loss: 97264.9453 - val_mae: 97264.9453
Epoch 3/100
250/250 [==============================] - 0s 1ms/step - loss: 97499.4453 - mae: 97499.4453 - val_loss: 93921.0391 - val_mae: 93921.0391
Epoch 4/100
250/250 [==============================] - 0s 1ms/step - loss: 92980.0391 - mae: 92980.0391 - val_loss: 88215.5156 - val_mae: 88215.5156
Epoch 5/100
250/250 [==============================] - 0s 1ms/step - loss: 86251.9453 - mae: 86251.9453 - val_loss: 80601.5938 - val_mae: 80601.5938
Epoch 6/100
250/250 [==============================] - 0s 1ms/step - loss: 77980.0625 - mae: 77980.0625 - val_loss: 72246.0938 - val_mae: 72246.0938
Epoch 7/100
250/250 [==============================] - 0s 1ms/step - loss: 69425.2188 - mae: 69425.2188 

In [34]:
## Load Tenserboard extendsion

%reload_ext tensorboard
model.save('Regrassion.h5')

c:\LearningWS\MachineLearning\mlenv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [35]:
## Load Tenserboard extendsion
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [36]:
%tensorboard --logdir reglogs/fit/

ERROR: Could not find `tensorboard`. Please ensure that your PATH
contains an executable `tensorboard` program, or explicitly specify
the path to a TensorBoard binary by setting the `TENSORBOARD_BINARY`
environment variable.

In [37]:
## Evaluate the model on the test set
test_loss, test_mae = model.evaluate(X_test, Y_test)
print(f"Test Loss: {test_loss}, Test MAE: {test_mae}")

63/63 [==============================] - 0s 908us/step - loss: 50344.5273 - mae: 50344.5273
Test Loss: 50344.52734375, Test MAE: 50344.52734375
